# Validation Test Simulation Setup - Capillary Wave

This notebook and the corresponding evaluation notebook (CapillaryWave_Evaluation.ipynb) are part of published results (cf. 7.1) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "..\..\..\src\L4-application\BoSSSpad\bin\Debug\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [2]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [3]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [4]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [5]:
wmg.Init("XNSEpaper_CapillaryWave", myBatch);

Project name is set to 'XNSEpaper_CapillaryWave'.
trying to run from backup database...
   Bkup Database dirs: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_CapillaryWave;
Selecting newest available backup: \\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_CapillaryWave
Opening existing database '\\dc3\userspace\smuda\hpccluster\bkup-2025Oct14_000000.XNSEpaper_CapillaryWave'.


In [6]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [7]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [9]:
var sessions = wmg.Sessions;
sessions

#0: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh8	4/14/2020 10:49:55 PM	b0bb4911...
#1: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh8	4/14/2020 10:51:49 PM	b2161c12...
#2: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh32*	4/18/2020 1:26:07 PM	7f9130d3...
#3: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh16	4/18/2020 12:45:23 PM	05b5ea49...
#4: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k2_mesh32	4/18/2020 12:45:40 PM	714dcbb2...
#5: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh16	4/18/2020 1:25:46 PM	06def218...
#6: XNSEpaper_CapillaryWave	CapillaryWave_La3e5_dtfix_convStudy_k3_mesh32_restart1	6/9/2020 1:36:19 PM	5e418fff...
#7: XNSEpaper_CapillaryWave	CapillaryWave_La3000_convStudy_k2_mesh8	3/18/2020 9:27:41 PM	7126f6b4...
#8: XNSEpaper_CapillaryWave	CapillaryWave_La120_convStudy_k2_mesh8	3/23/2020 9:34:58 PM	fcfecf85...
#9: XNSEpaper_CapillaryWave	CapillaryWa

## Grid Creation for Convergence Study

We consider the damped oscillations of a capillary wave initiated by a sinusoidal disturbance with a wavelength of $\lambda = 1$
and an initial amplitude height of $a_0 = 0.01 ≪ \lambda$, see Figure 3.

In [10]:
double L = 1;    // equals lambda
L

1

All physical settings are computed on three meshes with equidistant mesh sizes of $\lambda∕h = \{8, 16, 32 \}$. Comparing with the analytical
solution we follow the work of Popinet and set the computational domain to $\Omega = [0, \lambda] × [−3\lambda∕2, 3\lambda∕2]$. The lower
and upper boundaries are imposed by a wall boundary condition and the side boundaries are periodic in $x$-direction.

In [11]:
int[] Resolutions = new int[] { 8, 16, 32 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"CapillaryWave_{Res}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(0, L, Res + 1);
        double[] yNodes = GenericBlas.Linspace(-(3.0/2.0)*L, (3.0/2.0)*L, (3 * Res) * 1); 
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicX: true);
    
        grd.Name = GridName;
    
        grd.DefineEdgeTags(delegate (double[] X) {
            string ret = null;
            if (Math.Abs(X[1] + (3.0/2.0)*L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[1] - (3.0/2.0)*L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            return ret;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Grid Edge Tags changed.
An equivalent grid (ce24fae6-74f7-4ad6-af3e-816bb0335ef5) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (83eee32c-024e-4e70-b3aa-87d2f3a80cb6) is already present in the database -- the grid will not be saved.
Grid Edge Tags changed.
An equivalent grid (76cbcff9-86e9-4acf-985c-c1690470b4f9) is already present in the database -- the grid will not be saved.


## Initial Values

In [12]:
double A0 = L/100;
A0

0.01

In [13]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double L = 1.0;" + 
    "double A0 = L/100;" + 
    "double Phi(double[] X) { " + 
    "    return X[1] - A0 * Math.Sin(X[0] * 2.0 * Math.PI / L); " + 
    "}");

## Physical Settings

The physical properties in both phases are set equal. For such a setting, that is, small-amplitude limit and same kinematic viscosities $\nu = \mu∕\rho$, 
Prosperetti presents an analytical solution of the amplitude height $a(t)$ for the corresponding initial value problem. 
In order to verify against a representative range of physical regimes, from an overdamped oscillation (low $La$) to a
highly oscillatory behavior (high $La$), we investigate a study of the Laplace number defined as $La = \sigma \rho \lambda ∕ \mu 2$ . The study
consists of four values with $La = \{3, 1.2 ⋅ 10^2, 3 ⋅ 10^3, 3 ⋅ 10^5 \}$ and the corresponding physical properties are given in
Table 1.
  

In [14]:
public class PhysicalSettings {

    public double rho;               
    public double mu;
    public double sigma;

    public double[,] dts;
    public double t_end;

    public PhysicalSettings(string setup) {
        switch(setup) {
            case "3e5": 
            case "3e5_dtfix": {
                // Capillary wave: Laplace number La = 3e5: 
                rho   = 1e-3;
                mu    = 1e-5;
                sigma = 3e-2;
                dts   = new double[,] {{8e-4, 3e-4, 1e-4, 4e-5}, {5e-4, 2e-4, 7e-5, 2e-5}};
                t_end = 0.4;
                break;
            }
            case "3000": {
                // Capillary wave: Laplace number La = 3000: 
                rho   = 1e-3;
                mu    = 1e-4;
                sigma = 3e-2;
                dts   = new double[,] {{8e-4, 3e-4, 1e-4, 4e-5}, {5e-4, 2e-4, 7e-5, 2e-5}};
                t_end = 0.4;
                break;
            }
            case "120": {
                // Capillary wave: Laplace number La = 120: 
                rho   = 1e-3;
                mu    = 5e-4;
                sigma = 3e-2;
                dts   = new double[,] {{8e-4, 3e-4, 1e-4, 4e-5}, {5e-4, 2e-4, 7e-5, 2e-5}};
                t_end = 0.4;
                break;
            }
            case "3": {
                // Capillary wave: Laplace number La = 3: 
                rho   = 1e-3;
                mu    = 1e-3;
                sigma = 3e-3;
                dts   = new double[,] {{2e-3, 1e-3, 3e-4, 2e-4}, {1e-3, 6e-4, 2e-4, 8e-5}};
                t_end = 0.4;
                break;
            }
        }  
    }

}

## Control settings

The time step sizes on each mesh are set according to the capillary time step
restriction (43) with a spatial discretization of order $k = 2$.

In [15]:
string[] LaStudy = { "3", "120", "3000", "3e5", "3e5_dtfix" };
int[] pDegS = { 2, 3 };

In [16]:
public static int GetLogPeriod(double dt, double tend) {
    int NoTs = (int)(tend/dt); 

    if (NoTs > 100)
        return 10;

    return 1;
}

In [17]:
public static int GetSavePeriod(double dt, double tend) {
    int NoTs = (int)(tend/dt); 
    
    return NoTs / 100;  // no more than 100 restart points
}

In [18]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
for(int iDeg = 0; iDeg < pDegS.Length; iDeg++) {
foreach (string Lapl in LaStudy) {
    bool dt_fixed = false;    // smallest timestep is chosen for convergence study
    if (Lapl == "3e5_dtfix") {
        dt_fixed = true;
    } else {
        if (pDegS[iDeg] == 3)
            continue;
    }

    var physSet = new PhysicalSettings(Lapl);

for(int iGrd = 0; iGrd < Grids.Length; iGrd++) {
 
    if (pDegS[iDeg] == 3 && iGrd == Grids.Length - 1)   // we skip the finest simulation for now -> can be found in origin database
            continue;
    
    XNSE_Control C = new XNSE_Control();
    
    int pDeg = pDegS[iDeg];   
    var grd  = Grids[iGrd];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   
    

    double[] param = new double[4];
    param[0] = 1;        // wavenumber;
    param[1] = L;        // wavelength
    param[2] = A0;       // initial disturbance
    param[3] = 0.0;      // y-gravity
    C.AdditionalParameters = param;

    C.PhysicalParameters.rho_A = physSet.rho;
    C.PhysicalParameters.rho_B = physSet.rho;
    C.PhysicalParameters.mu_A  = physSet.mu;
    C.PhysicalParameters.mu_B  = physSet.mu;
    C.PhysicalParameters.Sigma = physSet.sigma;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    double dt       = (dt_fixed) ? physSet.dts[iDeg, 3] : physSet.dts[iDeg, iGrd];
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = physSet.t_end;
    C.NoOfTimesteps = (int)(physSet.t_end/dt); 
    
    C.saveperiod = GetSavePeriod(dt, physSet.t_end);

    C.PostprocessingModules.Add(new WaveLikeLogging(){ SolverStage = 2, LogPeriod = GetLogPeriod(dt, physSet.t_end) });
    
    C.SessionName = "CapillaryWave_La"+Lapl+"_convStudy_k" + pDeg + "_mesh" + Resolutions[iGrd];
    
    Controls.Add(C);
}
}
}

In [ ]:
foreach (var ctrl in Controls) {
    Console.WriteLine($"{ctrl.SessionName}");
}

In [21]:
int NoCtrls = Controls.Count();
NoCtrls

17

## Launch Jobs

In [22]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
}

Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh8	3/23/2020 9:35:37 PM	bdd58f86..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FinishedSuccessful);
Success: 1
Info: Found successful session "XNSEpaper_CapillaryWave	CapillaryWave_La3_convStudy_k2_mesh16	3/24/2020 2:29:40 PM	eed03bcf..." -- job is marked as successful, no further action.
No submission, because job status is: FinishedSuccessful
Deployments so far (1): (Job token: unknown, FinishedSuccessful 'unkown_DeploymentDirectory' @ MS HPC client  FDY-W

## Wait for Completion and Check Job Status

In [23]:
int NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress 
                                                                || job.Status == JobStatus.PendingInExecutionQueue
                                                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

0

In [24]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [25]:
int NoFailed = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

0

In [27]:
int NoSuccess = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

17

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [26]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [28]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");